In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from shapely.ops import polygonize
import json
import geopandas as gpd
import topojson as tp

In [ ]:
import warnings
# 필터링 실행 줄보다 '위쪽(먼저)'에 위치해야 함
warnings.filterwarnings("ignore", message=".*GeoSeries\\.notna.*", category=UserWarning)

In [ ]:
DATA_DIR = "/Library/WebServer/Documents/data"
file_path = [
    f"{DATA_DIR}/earth-topo.json",
    f"{DATA_DIR}/download/rtofs_glo_2ds_f000_ice.nc",
    f"{DATA_DIR}/download/rtofs_glo_2ds_f000_prog.nc"
]

1. earth-topo.json 로드 및 육지(Land) 폴리곤 생성
- earth-topo.json의 coastline_50m은 LineString이므로 폴리곤화(polygonize) 작업이 필요합니다.

In [ ]:
with open(file_path[0], "r", encoding="utf-8") as f:
    topo_data = json.load(f)

- TopoJSON -> GeoDataFrame 변환 (해상도에 따라 'coastline_50m' 또는 'coastline_110m' 선택)
- coastline_50m와 coastline_110m은 해안선(육지/해안 경계) 데이터이고, lakes_50m와 lakes_110m는 호수 데이터입니다.
- 육지 판정(contains / intersects 등) 목적이라면 해안선 레이어를 고르되,
선(LineString)인지 폴리곤(Polygon / MultiPolygon)인지 지오메트리 타입도 한 번 확인해 보세요.

In [ ]:
topo_data['objects'].keys()

In [ ]:
gdf_coast = tp.Topology(topo_data, object_name="coastline_50m").to_gdf()

- 해안선(LineString)들을 닫힌 다각형(Polygon, 즉 육지 영역)으로 변환
- earth-topo.json은 50m(1:50,000,000) 축척 데이터인 반면 RTOFS 글로벌 모델은 약 1/12°(약 8~9km) 해상도를 가집니다.
- 따라서 좁은 해협, 피오르드, 작은 섬 부근의 연안 격자는 50m 축척 폴리곤과 미세한 경계 차이로 인해 육지 내부로 판정될 수 있습니다.
- 연안 오차를 감안하려면 land_gdf.geometry = land_gdf.geometry.buffer(-0.05)처럼 육지 경계를 안쪽으로 살짝 축소하여 테스트하는 것이 좋습니다.

In [ ]:
land_polygons = list(polygonize(gdf_coast.geometry.union_all()))
land_gdf = gpd.GeoDataFrame(geometry=land_polygons, crs="EPSG:4326")


In [ ]:
# land_gdf.geometry = land_gdf.geometry.buffer(-0.05)
# 전 세계 미터 단위 투영좌표계(Equal Earth: ESRI:54035)로 변환 후 버퍼 적용
# (안쪽으로 줄이려면 음수 값, 바깥으로 넓히려면 양수 값 사용; 예: -5000m = -5km)
buffer_meters = -5000
land_proj = land_gdf.to_crs("ESRI:54035")
land_proj["geometry"] = land_proj.geometry.buffer(buffer_meters)

#  다시 경위도(EPSG:4326)로 복원 (빈/유효하지 않은 도형 제거)
land_gdf_4326 = land_proj.to_crs("EPSG:4326")
mask = land_gdf_4326.geometry.notna() & (~land_gdf_4326.geometry.is_empty)
land_gdf_4326 = land_gdf_4326.loc[mask].copy()

2. RTOFS NetCDF 파일 열기

In [ ]:
ds = xr.open_dataset(file_path[2])

- 확인하려는 변수명 지정 (예: 해빙 농도 'ice_coverage' 등 파일 내부 변수명에 맞게 수정)
- 변수 목록 확인:
```python
print(list(ds.data_vars))
```

In [ ]:
target_vars = list(ds.data_vars)
print(f"검사 대상 변수: {target_vars}")

- 첫 번째 데이터 변수 자동 선택 (필요시 직접 지정)

In [ ]:
target_var = target_vars[3]
da = ds[target_var].isel(MT=0) if "MT" in ds[target_var].dims else ds[target_var].squeeze()

In [ ]:
#- 위경도 변수 추출
lon = ds["Longitude"].values.flatten()
lon = np.where(lon >= 360, lon % 360, lon)
lat = ds["Latitude"].values.flatten()
val = da.values.flatten()

In [ ]:
#- 유효 데이터(NaN이 아닌 실제 값이 존재하는 격자점)만 필터링
valid_mask = np.isfinite(val)
valid_lons = lon[valid_mask]
valid_lats = lat[valid_mask]

print(f"검사 대상 변수: {target_var}")
print(f"전체 유효 데이터 격자 수: {len(valid_lons):,}개")

In [ ]:
#- 유효 데이터 좌표를 GeoDataFrame(Point)으로 변환 후 공간 조인(Spatial Join)
points_gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(valid_lons, valid_lats),
    crs="EPSG:4326"
)

In [ ]:
#- 육지(land_gdf) 내부에 포함된(within) 포인트 검색
points_in_land = gpd.sjoin(points_gdf, land_gdf_4326, how="inner", predicate="within")

In [ ]:
#- 결과 출력
land_data_count = len(points_in_land)
if land_data_count > 0:
    print(f"[결과] 육지(Land) 영역에 포함된 {target_var} 데이터가 존재합니다! (총 {land_data_count:,}개)")
    print("샘플 좌표 (경도, 위도):")
    print(points_in_land.geometry.head(10))
else:
    print("[결과] 육지(Land) 영역에 포함된 데이터가 없습니다. (모두 해양 영역에 위치)")